# Week 1 Cold-Start Prediction — 2025 Week 1

Week 1 is the hardest week to predict: every in-season rolling feature (3/5-week averages, target share, opportunity share, ...) is zero because there are no prior games. The model must lean on the information that *does* exist before kickoff:

- **Previous-season carryover** — `prev_season_ppg`, `prev_season_games` (a player's established level)
- **Vegas** — `implied_total`, `team_spread`, `team_win_prob` (the market's game script forecast)
- **Depth chart** — `depth_chart_rank` (role entering the season)
- **Game environment** — home/away, dome, temperature, wind

**Setup:** train on 2020–2024 (fantasy weeks 1–17 only), early-stop on 2024 Week 1 (the most recent week-1-like distribution), predict **2025 Week 1**. The model sees `week` as a feature, so it learns from five historical week 1s how to behave when rolling features are all zero.

Expect a higher MAE than mid-season weeks — that is the price of having no current-season signal, and it is unavoidable for any model.

*Production caveat:* rows here are players who actually recorded a stat in 2025 Week 1 (the Gold table is built from play-by-play). A live pre-game system would instead build the slate from the schedule + roster + depth chart; the features and model are identical.

In [0]:
# Install modeling dependencies
%pip install xgboost scikit-learn matplotlib

## Load Gold Table and Preprocess

Identical leakage guard to the walk-forward notebook: drop identifiers and every same-week outcome column, keep only pre-game features, one-hot encode position.

In [0]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Load gold table from Unity Catalog Delta table
if 'gold_df' not in globals():
    gold_df = spark.table("fantasy_football.gold.player_weeks").toPandas()

# Fantasy season only (weeks 1-17)
gold_df = gold_df[gold_df['week'] <= 17]

TARGET = 'fantasy_points_ppr'

eval_meta = gold_df[['player_id', 'player_name', 'recent_team', 'position', 'season', 'week']].copy()

df = gold_df.copy()

identifier_cols = [
    'player_id', 'player_name', 'recent_team',
    'opponent', 'starting_qb_id', 'gameday',
]
df = df.drop(columns=[c for c in identifier_cols if c in df.columns])

# Same-week outcome columns would leak the answer
same_week_outcome_cols = [
    'pass_attempts', 'completions', 'passing_yards', 'passing_tds', 'interceptions',
    'rush_attempts', 'rushing_yards', 'rushing_tds',
    'targets', 'receptions', 'receiving_yards', 'receiving_tds',
    'player_opportunities', 'team_total_opportunities', 'opportunity_share',
    'hvt_carries', 'hvt_targets', 'total_hvts',
    'team_pass_attempts', 'target_share',
    'player_air_yards', 'team_air_yards', 'air_yards_share',
    'wopr', 'snap_share',
]
df = df.drop(columns=[c for c in same_week_outcome_cols if c in df.columns])

df['position'] = eval_meta['position']
df = pd.get_dummies(df.fillna(0), columns=['position'], prefix='pos')
feature_cols = [c for c in df.columns if c != TARGET]

print(f"Modeling dataset: {df.shape[0]:,} rows x {len(feature_cols)} features")
print(f"Seasons: {sorted(eval_meta['season'].unique())}")

## Train and Predict 2025 Week 1

- **Train**: every 2020–2024 row except 2024 Week 1 (~five full seasons, all weeks — mid-season rows teach the model what the features mean, historical week 1s teach it the cold-start regime)
- **Validation (early stopping)**: 2024 Week 1 — stops boosting when error on real week-1 data stops improving, which directly fights overfitting to mid-season patterns
- **Test**: 2025 Week 1, never seen during training or stopping

Two naive baselines for context: predicting each player's previous-season PPG, and predicting the training-set mean.

In [0]:
# ============================================================================
# TRAIN (2020-2024) -> EARLY-STOP (2024 wk1) -> PREDICT (2025 wk1)
# ============================================================================
TEST_SEASON = int(eval_meta['season'].max())
VAL_SEASON = TEST_SEASON - 1

test_mask = (eval_meta['season'] == TEST_SEASON) & (eval_meta['week'] == 1)
val_mask = (eval_meta['season'] == VAL_SEASON) & (eval_meta['week'] == 1)
train_mask = (eval_meta['season'] < TEST_SEASON) & ~val_mask

print(f"Train: {train_mask.sum():,} rows | Val (early stop): {val_mask.sum():,} rows "
      f"| Test ({TEST_SEASON} wk1): {test_mask.sum():,} rows")

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)
model.fit(
    df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET],
    eval_set=[(df.loc[val_mask, feature_cols], df.loc[val_mask, TARGET])],
    verbose=False,
)
print(f"Early stopping picked {model.best_iteration} boosting rounds")

preds = np.clip(model.predict(df.loc[test_mask, feature_cols]), 0, None)

results = eval_meta.loc[test_mask, ['player_name', 'recent_team', 'position']].copy()
results['predicted'] = preds.round(1)
results['actual'] = df.loc[test_mask, TARGET].values
results['abs_error'] = (results['actual'] - results['predicted']).abs()

# --- Evaluation ---
model_mae = results['abs_error'].mean()
naive_prev = df.loc[test_mask, 'prev_season_ppg'].clip(lower=0)
naive_prev_mae = (results['actual'] - naive_prev.values).abs().mean()
naive_mean_mae = (results['actual'] - df.loc[train_mask, TARGET].mean()).abs().mean()

print(f"\n2025 WEEK 1 MAE:")
print(f"  XGBoost cold-start model : {model_mae:.2f}")
print(f"  Naive: prev-season PPG   : {naive_prev_mae:.2f}")
print(f"  Naive: training mean     : {naive_mean_mae:.2f}")

print("\nMAE by position:")
print(results.groupby('position')['abs_error'].agg(['mean', 'count'])
      .reindex(['QB', 'RB', 'WR', 'TE']).round(2).to_string())

## Predicted Rankings vs Reality

Top 25 projected players for 2025 Week 1, plus the biggest misses in both directions — useful for judging whether errors come from usage surprises (injuries, surprise starters) or from the model itself.

In [0]:
# ============================================================================
# TOP-25 PROJECTIONS AND BIGGEST MISSES
# ============================================================================
display_cols = ['player_name', 'recent_team', 'position', 'predicted', 'actual', 'abs_error']

print(f"TOP 25 PROJECTED PLAYERS — {TEST_SEASON} WEEK 1")
display(results.sort_values('predicted', ascending=False).head(25)[display_cols]
        .reset_index(drop=True))

print("\nBIGGEST OVER-PROJECTIONS (predicted high, scored low):")
over = results.assign(miss=results['predicted'] - results['actual'])
display(over.sort_values('miss', ascending=False).head(5)[display_cols].reset_index(drop=True))

print("\nBIGGEST UNDER-PROJECTIONS (breakouts the model missed):")
display(over.sort_values('miss').head(5)[display_cols].reset_index(drop=True))

## What Drives Week 1 Predictions

Feature importance for the cold-start model. Expect previous-season carryover, Vegas lines, and depth chart to dominate, since all in-season rolling features are zero for week 1 rows.

In [0]:
# ============================================================================
# FEATURE IMPORTANCE (gain)
# ============================================================================
import matplotlib.pyplot as plt

importance = (
    pd.Series(model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .head(15)
)
print("Top 15 features (gain importance):")
print(importance.round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 6))
importance.iloc[::-1].plot.barh(ax=ax)
ax.set_title(f'Week 1 Cold-Start Model — Top 15 Features ({TEST_SEASON} Week 1)')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.show()